# 02 — Exploratory Analysis & Choropleth Maps
Exploratory analysis of 2021/2022 municipal election results, including choropleth maps of winning parties per municipality.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('Resultados_CM_2022.csv', encoding='latin1')
print(df.head())

##### Retirar colunas que não interessam

In [ ]:
df = df.drop(columns=['COD'])
df = df.drop(columns=['ORG'])

##### Adicionar coluna com soma das colunas

In [ ]:
df["sum_columns"] = df.select_dtypes(include=[np.number]).apply(np.sum, axis=1).round(2)

In [ ]:
histogram = df["sum_columns"].hist(bins=20, edgecolor='black')
plt.title('Histogram of Sum of Numeric Columns')

In [ ]:
boxplot = df.boxplot(column='sum_columns')
plt.title('Boxplot of Sum of Numeric Columns')
plt.show()

In [ ]:
sample_df = df[['CONC','sum_columns']]

In [ ]:
#trocar "missing values" por 0
df = df.fillna(0)

df['PSD/CDS'] = df['CDS-PP'] + df['PPD/PSD']
df = df.drop(columns=['CDS-PP', 'PPD/PSD'])

In [ ]:
df = df.rename(columns=lambda x: f"{x}_2021" if x != "CONC" else x)


In [ ]:
df.to_csv('resultados_CM_2021.csv', sep=';', index=False)

In [ ]:
# Cria uma cópia sem a coluna sum_columns
df_clean = df.drop(columns=['sum_columns'])

# Aplica novamente:
max_partido = df_clean.iloc[:, 3:].idxmax(axis=1)
max_percentagem = df_clean.iloc[:, 3:].max(axis=1)

resultado = pd.DataFrame({
    'CONC': df_clean['CONC'],
    'Percentagem_Max': max_percentagem,
    'Partido_Vencedor': max_partido
})

In [ ]:
resultado.to_csv('resultado_partido_vencedor_cm_2022.csv', index=False, sep=';')

In [ ]:
# Define o dicionário de cores
cores_partido = {
    'PS': 'purple',
    'PSD/CDS': 'orange',
    'B.E.': 'blue',
    'CDU': 'green',
    'IL': 'lightblue',
    'CH': 'black',
    'PCP-PEV': 'red',
}

# Cria uma coluna com as cores
resultado['cor'] = resultado['Partido_Vencedor'].map(cores_partido)
# Atribui 'grey' (cinzento) onde ficou NaN (partidos não listados)
resultado['cor'] = resultado['cor'].fillna('grey')


In [ ]:
import geopandas as gpd

# Substitui 'mapa_portugal.shp' pelo nome do teu ficheiro .shp
gdf = gpd.read_file('concelhos.shp')

# Verifica o conteúdo
print(gdf.head())

In [ ]:
import unicodedata

def remover_acentos(texto):
    if isinstance(texto, str):
        return ''.join(
            c for c in unicodedata.normalize('NFD', texto)
            if unicodedata.category(c) != 'Mn'
        )
    return texto

gdf['CONC'] = gdf['NAME_2'].apply(remover_acentos)

In [ ]:
gdf['concelho_std'] = gdf['CONC'].str.upper().str.strip()
resultado['CONC_std'] = resultado['CONC'].str.upper().str.strip()

In [ ]:
gdf_merged = gdf.merge(resultado, left_on='concelho_std', right_on='CONC_std', how='left')


In [ ]:
cores_partido = {
    'PS': 'purple',
    'PSD/CDS': 'orange',
    'B.E.': 'blue',
    'CDU': 'green',
    'IL': 'lightblue',
    'CH': 'black',
    'PCP-PEV': 'red',
}

gdf_merged['cor'] = gdf_merged['Partido_Vencedor'].map(cores_partido).fillna('grey')

In [ ]:
# devolver os unique values de 'Name_1'
print(gdf['NAME_1'].unique())

In [ ]:
def classificar_regiao(name1):
    if name1 == 'Azores':
        return 'Acores'
    elif name1 == 'Madeira':
        return 'Madeira'
    else:
        return 'Continente'

gdf_merged['Regiao'] = gdf_merged['NAME_1'].apply(classificar_regiao)

In [ ]:
import matplotlib.pyplot as plt

# Portugal Continental
gdf_continental = gdf_merged[gdf_merged['Regiao'] == 'Continente']
fig, ax = plt.subplots(figsize=(10, 10))
gdf_continental.plot(color=gdf_continental['cor'], edgecolor='black', ax=ax)
ax.set_title('Portugal Continental')
ax.axis('off')
plt.show()

# Açores
gdf_acores = gdf_merged[gdf_merged['Regiao'] == 'Acores']
fig, ax = plt.subplots(figsize=(10, 10))
gdf_acores.plot(color=gdf_acores['cor'], edgecolor='black', ax=ax)
ax.set_title('Acores')
ax.axis('off')
plt.show()

# Madeira
gdf_madeira = gdf_merged[gdf_merged['Regiao'] == 'Madeira']
fig, ax = plt.subplots(figsize=(10, 10))
gdf_madeira.plot(color=gdf_madeira['cor'], edgecolor='black', ax=ax)
ax.set_title('Madeira')
ax.axis('off')
plt.show()
